# `QUAX`: Accelerated Quantum Information

This notebook demonstrates the `quax` module which allows for hardware-accelerated and differentiable calculations using quantum states, gates and superoperators.

### Quantum Objects: States, operators and superoperators

Quantum objects are sorted into three categories: `State`, `Operator` and `Superoperator`


`State` includes `StateVector` and `DensityMatrix`

`Operator` includes `Unitary` and `Kraus`

`Superoperator` includes `SuperOp`, `Choi`, `PauliLiouville`, `Chi`

### Operator syntax

For all objects, the following operations are defined:

- `@ `   : composition / action
- `| `   : tensor product
- `* `   : scalar multiplication
- `**`  : powers
- `- `   : negation

Not all operations are defined for all combinations of objects. For example, while we can tensor product any two `Superoperator` objects, we can't tensor product a `Choi` and `DensityMatrix` - this doesn't make sense.

When we operator with two compatible objects of different types, for example `Choi @ SuperOp`, the result will generally be the type of the right object - so in this case a `Choi`. Only the forward `__matmul__` is defined.

#### Examples

Apply a `Unitary` to a `StateVector`

```python
initial_psi = zero_state_vector(2)
psi = CZ @ initial_psi
```

Apply a sequence of `Unitary` to a `DensityMatrix`

```python
mixed_rho = mixed_state_matrix(2)
rho = CZ @ (RX(jnp.pi/2) | RZ(jnp.pi / 2)) @ mixed_rho
```

Apply a `Unitary` to an `DensityMatrix` ensemble. Note here that the ensemble indices are leading, the final two dimensions should always be the density matrix. The dims remain the same - one requirement of this kind of batched operations is that the density matrices are of the same size.

```python
mixed_rhos = DensityMatrix(
    data=jnp.array([mixed_state_matrix(2).data, mixed_state_matrix(2).data]),
    dims=(2, 2),
)
rhos = CZ @ mixed_rhos
```

### Just-in-time compilation (JIT)

JIT is an essential part of Jax. Not only is it essential for achieving high performance, but it's a hard requirement for using jax's unique capabilities like `vmap` and `grad`.

JIT comes with strings attached - specifically, the dimensions of the calculation must be defined beforehand. Operators, states and superoperators have various dimensions depending both on the number of qubits and their dimension (qutrits, quarts etc.). The dimensions are always carried by the states and operators, and JITs will automatically be specific to those dimensions which are static args for the functions. Changing the dimensions of operator or state will result in a new JIT.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import jax
import jax.numpy as jnp

import quax as qx
from quax.gates import CZ, ISWAP, RX, RZ

## States

Let's start by initializing some states.

In [3]:
initial_psi = qx.zero_state_vector(3)
initial_rho = qx.zero_state_matrix(3)
mixed_rho = qx.mixed_state_matrix(2)

In [4]:
mixed_rho

DensityMatrix(data=Array([[[[0.25+0.j, 0.  +0.j],
         [0.  +0.j, 0.  +0.j]],

        [[0.  +0.j, 0.25+0.j],
         [0.  +0.j, 0.  +0.j]]],


       [[[0.  +0.j, 0.  +0.j],
         [0.25+0.j, 0.  +0.j]],

        [[0.  +0.j, 0.  +0.j],
         [0.  +0.j, 0.25+0.j]]]], dtype=complex128), num_ensemble_dims=0)

## Operators

We can also use the standard pyquil gates

In [5]:
RX(jnp.pi / 2)

Unitary(data=Array([[0.70710678+0.j        , 0.        -0.70710678j],
       [0.        -0.70710678j, 0.70710678+0.j        ]],      dtype=complex128), num_ensemble_dims=0)

Unitaries can be composed

In [6]:
CZ @ ISWAP

Unitary(data=Array([[[[ 1.+0.j,  0.+0.j],
         [ 0.+0.j,  0.+0.j]],

        [[ 0.+0.j,  0.+0.j],
         [ 0.+1.j,  0.+0.j]]],


       [[[ 0.+0.j,  0.+1.j],
         [ 0.+0.j,  0.+0.j]],

        [[ 0.+0.j,  0.+0.j],
         [ 0.+0.j, -1.+0.j]]]], dtype=complex128), num_ensemble_dims=0)

They can also be tensored

In [7]:
RX(jnp.pi / 2) | RZ(jnp.pi / 4)

Unitary(data=Array([[[[ 0.65328148-0.27059805j,  0.        +0.j        ],
         [-0.27059805-0.65328148j,  0.        +0.j        ]],

        [[ 0.        +0.j        ,  0.65328148+0.27059805j],
         [ 0.        +0.j        ,  0.27059805-0.65328148j]]],


       [[[-0.27059805-0.65328148j,  0.        +0.j        ],
         [ 0.65328148-0.27059805j,  0.        +0.j        ]],

        [[ 0.        +0.j        ,  0.27059805-0.65328148j],
         [ 0.        +0.j        ,  0.65328148+0.27059805j]]]],      dtype=complex128), num_ensemble_dims=0)

They can be multiplied. Note that multiplying a Unitary by a complex number not equal to 1 will result in a Kraus

In [8]:
1.0 * CZ

Unitary(data=Array([[[[ 1.+0.j,  0.+0.j],
         [ 0.+0.j,  0.+0.j]],

        [[ 0.+0.j,  1.+0.j],
         [ 0.+0.j,  0.+0.j]]],


       [[[ 0.+0.j,  0.+0.j],
         [ 1.+0.j,  0.+0.j]],

        [[ 0.+0.j,  0.+0.j],
         [ 0.+0.j, -1.+0.j]]]], dtype=complex128), num_ensemble_dims=0)

In [9]:
0.5 * CZ

Kraus(data=Array([[[[ 0.5+0.j,  0. +0.j],
         [ 0. +0.j,  0. +0.j]],

        [[ 0. +0.j,  0.5+0.j],
         [ 0. +0.j,  0. +0.j]]],


       [[[ 0. +0.j,  0. +0.j],
         [ 0.5+0.j,  0. +0.j]],

        [[ 0. +0.j,  0. +0.j],
         [ 0. +0.j, -0.5+0.j]]]], dtype=complex128), num_ensemble_dims=0)

They can be fractionally powered

In [10]:
CZ ** (0.5)

Unitary(data=Array([[[[1.000000e+00+0.j, 0.000000e+00+0.j],
         [0.000000e+00+0.j, 0.000000e+00+0.j]],

        [[0.000000e+00+0.j, 1.000000e+00+0.j],
         [0.000000e+00+0.j, 0.000000e+00+0.j]]],


       [[[0.000000e+00+0.j, 0.000000e+00+0.j],
         [1.000000e+00+0.j, 0.000000e+00+0.j]],

        [[0.000000e+00+0.j, 0.000000e+00+0.j],
         [0.000000e+00+0.j, 6.123234e-17+1.j]]]], dtype=complex128), num_ensemble_dims=0)

Operations can also be broadcast

### We can act on states using unitary matrices

Naturally, we can also apply a unitary matrix to a state

In [11]:
initial_psi = qx.zero_state_vector(2)
qx.random_state_vector(dims=(2,), key=jax.random.PRNGKey(0))
CZ @ initial_psi

StateVector(data=Array([[1.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j]], dtype=complex128), num_ensemble_dims=0)

We can also act on mixed states.

In [12]:
initial_rho = qx.mixed_state_matrix(2)

CZ @ initial_rho

DensityMatrix(data=Array([[[[0.25+0.j, 0.  +0.j],
         [0.  +0.j, 0.  +0.j]],

        [[0.  +0.j, 0.25+0.j],
         [0.  +0.j, 0.  +0.j]]],


       [[[0.  +0.j, 0.  +0.j],
         [0.25+0.j, 0.  +0.j]],

        [[0.  +0.j, 0.  +0.j],
         [0.  +0.j, 0.25+0.j]]]], dtype=complex128), num_ensemble_dims=0)

Or even an ensemble of states

In [13]:
size = (3, 5)

rhos = qx.random_density_matrix(rank=2, dims=(2, 2), key=jax.random.PRNGKey(0), size=size)
rho_outs = CZ @ rhos
print(rho_outs)

DensityMatrix(dims=(2, 2), ensemble_size=(3, 5), shape=(3, 5, 2, 2, 2, 2))


## Superoperators

Superoperators describe noisy operations. We can promote a unitary to a superoperator.

In [14]:
choi = qx.random_choi_BCSZ(dims=((2, 2, 2), (2, 2, 2)), rank=4, key=jax.random.key(1))

In [15]:
S = qx.unitary_to_superop(CZ)
print(S)

SuperOp(dims=((2, 2), (2, 2)), shape=(2, 2, 2, 2, 2, 2, 2, 2))


### Common channels

We can construct some common superoperators such as bit flips, amplitude damping and depolarizing.

In [16]:
S = qx.depolarizing_channel_superoperator(0.1, 1)
print(S)

SuperOp(dims=((2,), (2,)), shape=(2, 2, 2, 2))


### Conversions

In [17]:
C = qx.superop_to_choi(S)
print(C)

P = qx.superop_to_pauli_liouville(S)
print(P)

Choi(dims=((2,), (2,)), shape=(2, 2, 2, 2))
PauliLiouville(dims=((2,), (2,)), shape=(2, 2, 2, 2))


### Composing channels

We can compose superoperators using the `@` symbol

In [18]:
print(S @ S)

SuperOp(dims=((2,), (2,)), shape=(2, 2, 2, 2))


We can also compose with Superoperators in different forms. By convention, the output type will be that of the lefthand operator.

Ie,

`SuperOp @ Choi -> SuperOp`

`Choi @ SuperOp -> Choi`

In [19]:
S @ C

Choi(data=Array([[[[0.905+0.j, 0.   +0.j],
         [0.   +0.j, 0.81 +0.j]],

        [[0.   +0.j, 0.095+0.j],
         [0.   +0.j, 0.   +0.j]]],


       [[[0.   +0.j, 0.   +0.j],
         [0.095+0.j, 0.   +0.j]],

        [[0.81 +0.j, 0.   +0.j],
         [0.   +0.j, 0.905+0.j]]]], dtype=complex128), num_ensemble_dims=0)

In [20]:
C @ S

SuperOp(data=Array([[[[0.905+0.j, 0.   +0.j],
         [0.   +0.j, 0.095+0.j]],

        [[0.   +0.j, 0.81 +0.j],
         [0.   +0.j, 0.   +0.j]]],


       [[[0.   +0.j, 0.   +0.j],
         [0.81 +0.j, 0.   +0.j]],

        [[0.095+0.j, 0.   +0.j],
         [0.   +0.j, 0.905+0.j]]]], dtype=complex128), num_ensemble_dims=0)

In [21]:
P @ S @ C

Choi(data=Array([[[[0.8645+0.j, 0.    +0.j],
         [0.    +0.j, 0.729 +0.j]],

        [[0.    +0.j, 0.1355+0.j],
         [0.    +0.j, 0.    +0.j]]],


       [[[0.    +0.j, 0.    +0.j],
         [0.1355+0.j, 0.    +0.j]],

        [[0.729 +0.j, 0.    +0.j],
         [0.    +0.j, 0.8645+0.j]]]], dtype=complex128), num_ensemble_dims=0)

### Tensoring Channels

Similarly, we can tensor channels

In [22]:
PxP = P | P
print(PxP)

PauliLiouville(dims=((2, 2), (2, 2)), shape=(2, 2, 2, 2, 2, 2, 2, 2))


In [23]:
PxCxS = P | C | S
print(PxCxS)

SuperOp(dims=((2, 2, 2), (2, 2, 2)), shape=(2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2))


## Distance Metrics

### State Fidelity

In [24]:
rho = qx.zero_state_matrix(2)
sigma = qx.mixed_state_matrix(2)

print(f"fidelity(𝜌, 𝜌) = {qx.fidelity(rho, rho):.2f}")
print(f"fidelity(𝜌, 𝜎) = {qx.fidelity(rho, sigma):.2f}")
print(f"fidelity(𝜎, 𝜎) = {qx.fidelity(sigma, sigma):.2f}")

fidelity(𝜌, 𝜌) = 1.00
fidelity(𝜌, 𝜎) = 0.25
fidelity(𝜎, 𝜎) = 1.00


In [25]:
choi = qx.unitary_to_choi(RX(jnp.pi))

print(f"Process fidelity (self): {qx.process_fidelity(choi, choi):.2f}")

print(f"Process fidelity (identity): {qx.process_fidelity(choi):.2f}")

Process fidelity (self): 1.00
Process fidelity (identity): 0.00


## Random Operators and Ensembles

Random operators have various uses in quantum information. We provide common ensembles and distributions of operators.

`random_unitary`: Sample random unitaries from the Haar measure.

In [26]:
key = jax.random.key(4258)
num_unitaries = (80, 22, 64)
dims = ((2,), (2,))
unitaries = qx.random_unitary(dims=dims, key=key, size=num_unitaries)

In [27]:
qx.is_two_design(unitaries, atol=1e-2)

Array(True, dtype=bool)

We can check properties of the ensemble: For example it's 1-design and 2-design properties

In [28]:
qx.is_two_design(qx.ensembles.CLIFFORD_ENSEMBLE, atol=1e-2)

Array(True, dtype=bool)

In [29]:
qx.is_two_design(qx.ensembles.TETRAHEDRAL_ENSEMBLE, atol=1e-2)

Array(True, dtype=bool)

## Simulate a program

Here, we'll demonstrate how to apply superoperators to a larger density matrix. This enables the simulation of programs.

In [33]:
rho = qx.zero_state_matrix(2)

# Generate some superoperators
sx = qx.unitary_to_superop(qx.gates.RX(jnp.pi / 2))
cz = qx.unitary_to_superop(qx.gates.CZ)

rho = qx.targeted_apply_superop(sx, rho, (0,))
rho = qx.targeted_apply_superop(sx, rho, (1,))
rho = qx.targeted_apply_superop(cz, rho, (0, 1))
rho.matrix

Array([[0.25+0.j  , 0.  +0.25j, 0.  +0.25j, 0.25+0.j  ],
       [0.  -0.25j, 0.25+0.j  , 0.25+0.j  , 0.  -0.25j],
       [0.  -0.25j, 0.25+0.j  , 0.25+0.j  , 0.  -0.25j],
       [0.25+0.j  , 0.  +0.25j, 0.  +0.25j, 0.25+0.j  ]],      dtype=complex128)